In [6]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [7]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [8]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = MODEL_GPT
openai = OpenAI()

API key looks good so far


In [9]:
# set up environment

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# here is the question; type over this to ask something new

# question = """
# Please explain what this code does and why:
# yield from {book.get("author") for book in books if book.get("author")}
# """

In [12]:
#system prompt

system_prompt = """
You are assistant or support integration engineer for Juspay Technologies. Who explains the question given to a client 
in an easy language so that someone who is a complete beginner can also easily understand it to integrate Juspay
"""

In [13]:
links = fetch_website_links("https://juspay.io/in/docs/")
links

['#',
 'https://juspay.io/integrations',
 '/dashboard/docs',
 '/api-reference/docs',
 '/in/docs/hyper-checkout/overview',
 '/in/docs/ec-headless/android/overview/integration-architecture',
 '/in/docs/ec-api/docs/overview/integration-architecture',
 '/in/docs/hyper-credit/web/overview/integration-architecture',
 '/in/docs/payment-form/docs/overview/create-payment-form',
 '/in/docs/lotuspay/web/overview/introduction',
 '/in/docs/upi-tpap-sdk/android/overview/integration-architecture',
 '/in/docs/upi-plugin-sdk/android/overview/integration-architecture',
 '/in/docs/payout/docs/overview/introduction',
 '/in/docs/jusbiz/docs/overview/introduction',
 '/product-summary/docs/product-summary/overview',
 '/payment-locking/docs/payment-locking/overview',
 '/quickpay-integration/docs/quick-pay/overview',
 '/outages/docs/outages/overview',
 '/cred-pay/docs/cred-pay/overview',
 '/surcharge/docs/surcharge/overview',
 '/mweb-intent/docs/upi-intent-on-mweb/overview',
 '/upi-autopay/docs/upi-autopay/ove

In [14]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant for the question from the user to help in the integration.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [15]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant for the question from the user to help in the integration, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [16]:
print(get_links_user_prompt("https://juspay.io/in/docs/"))


Here is the list of links on the website https://juspay.io/in/docs/ -
Please decide which of these are relevant for the question from the user to help in the integration, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#
https://juspay.io/integrations
/dashboard/docs
/api-reference/docs
/in/docs/hyper-checkout/overview
/in/docs/ec-headless/android/overview/integration-architecture
/in/docs/ec-api/docs/overview/integration-architecture
/in/docs/hyper-credit/web/overview/integration-architecture
/in/docs/payment-form/docs/overview/create-payment-form
/in/docs/lotuspay/web/overview/introduction
/in/docs/upi-tpap-sdk/android/overview/integration-architecture
/in/docs/upi-plugin-sdk/android/overview/integration-architecture
/in/docs/payout/docs/overview/introduction
/in/docs/jusbiz/docs/overview/introduction
/product-summary/docs/product-summary/overview
/payment-locking/docs/payment-locking/ove

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
#system prompt

assistant_system_prompt = """
You are assistant or support integration engineer for Juspay Technologies. Who explains the question given to a client 
in an easy language so that someone who is a complete beginner can also easily understand it to integrate Juspay
"""

In [20]:
def question_prompt(question):
    question_string = """Please explain this Question: """+ question
    # yield from {book.get("author") for book in books if book.get("author")}
    return question_string

In [21]:
def question_prompt(question, url):
    user_prompt = f"""
Please help: {question}
Here are the contents of its landing page and other relevant pages;
use this information to help a integration questions.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
# Get gpt-4o-mini to answer, with streaming

def ask_theia(question, url):
    # question = input("Enter your Question")
    # url = input ("Enter the doc link:")
    # print(f"Question: {question}")
    # print(f"URL: {url}")
    stream = openai.chat.completions.create(
        model= MODEL_GPT,
        messages=[
            {"role": "system", "content": assistant_system_prompt},
            {"role": "user", "content": question_prompt(question, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
ask_theia("What is the API that needs to be called for creating the order?", "https://juspay.io/in/docs/")

Selecting relevant links for https://juspay.io/in/docs/ by calling gpt-4o-mini
Found 11 relevant links


To create an order using Juspay, you will need to call the **Express Checkout API**. This API allows your application or website to collect payments. Here’s a step-by-step explanation of what you need to do:

1. **Understand the API**: An API (Application Programming Interface) is like a waiter in a restaurant. You tell the waiter what you want (your order), and the waiter (the API) communicates with the kitchen (the payment processing system) to get your order processed.

2. **Set Up Your Environment**: If you haven't already, you'll need to set up your development environment. This usually includes signing up for a Juspay account and getting API keys—these are like a password that lets you access Juspay’s services securely.

3. **Create the Order**:
   - Using the Express Checkout API, you will send a request containing details of the order you want to create. This typically includes information like the items being purchased, the price, and any customer details needed.
   - This action is like writing down your order and handing it to the waiter.

4. **Receive a Response**: After you send the request, the API will respond with a confirmation of the order or an error message if something went wrong. It’s similar to getting feedback from the waiter confirming your food is on the way.

5. **Handle the Payment**: Once the order is created, you can proceed with the payment process through Juspay’s interface or SDK, depending on your chosen integration method.

For more detailed steps, you can refer to the **API reference** section in the Juspay Developer Docs, where you’ll find specific information on how to format your requests and what responses to expect.

In summary, to create an order, focus on using the **Express Checkout API**. Make sure your app or website is set up to send the necessary information, and you’ll be on your way to integrating payments through Juspay! If you need any further help or specific examples, feel free to ask!

In [25]:
def question_prompt(question):
    question_string = """Please explain this Question: """+ question
    # yield from {book.get("author") for book in books if book.get("author")}
    return question_string

In [ ]:
system_prompt = """ You are a Technical Assistant who explains the question given to a user 
in an easy language so that someone who is a complete beginner can also easily understand it"""

In [ ]:
# Get openai to answer, with streaming
def ask_question():
    question = input("Enter your Question")
    print(f"Question: {question}")
    stream = ollama.chat.completions.create(model=MODEL_LLAMA,messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question_prompt(question)}
            ],
            stream=True
        )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [31]:
ask_question()

Question: How to improve my batting skill


Improving your batting skills can be broken down into several steps. Let's start with the basics.

**What does Good Batting Mean?**

Good batting means hitting the ball with a bat in a cricket game or any other sport that involves hitting a ball. It requires timing, technique, and practice.

**Key Skills to Improve:**

To improve your batting skill, focus on these three essential skills:

1. **Eye Care**: The ability to locate the ball accurately before it hits the bat is crucial for good batting.
2. **Footwork**: Shuffling, running, or standing still – the way you position yourself in front of the wicket (the batsman) affects your chances of scoring runs.
3. **Technique**: This includes how you grip the bat, swing, and hit the ball.

**Easy Steps to Improve Batting Skill:**

Here are some easy steps to improve your batting skill:

**Step 1: Warm-up and Relaxation**
Begin by doing simple exercises like stretching or jogging in the field to get familiar with the balls. Take a few deep breaths to relax and focus on your game.

**Step 2: Watch and Practice Balls**
Watch professionals or seniors batsmen practice their skills. Practice hitting different types of balls (fast, slow, straight, curved) with a ball and bat combo. Get comfortable with different lengths and angles.

**Step 3: Learn Footwork Patterns**
Develop simple footwork patterns, such as the 'step-and-ball-drive' or 'square-cut-drives.' These will help you develop your timing and positioning in front of the wicket.

**Step 4: Timing Tricks**
Focus on reading the bowler's action (the speed and style) to anticipate when the ball is coming. The timing trick is all about anticipation!

**Step 5: Join a Batting Session**
Practice with friends, family, or as part of a team practice session. Regular batting practices will help you apply these steps.

**Tips and Words of Wisdom**

* Practice regularly, patiently.
* Keep trying new skills gradually – your brain takes time!
* Visualize yourself hitting runs.
* Don’t get discouraged if you miss the ball; every error is a learning opportunity!

By practicing these simple steps consistently, following expert advice, you'll begin to see improvements in your batting performance. Enjoy playing and scoring more confidently!